# Modelos Embedding

### Empezamos cargando los modelos

usaremos estos modelos:
    "bge-m3": "BAAI/bge-m3",
    "Qwen": "Qwen/Qwen3-VL-Embedding-2B",
    "embeddinggemma-300m": "google/embeddinggemma-300m",
    "embeddinggemma-2": "google/embeddinggemma-2",
    "sentence-transformers": "sentence-transformers/all-MiniLM-L6-v2"


In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

# Carga las variables de entorno desde .env
load_dotenv()
hf_token = os.getenv("HF_TOKEN") or None

# Carpeta donde se guardarán los modelos
BASE_DIR = Path("/home/g5/reto_1_2-2627-Rag-Benchmark/Modelos/Embedding")
BASE_DIR.mkdir(parents=True, exist_ok=True)

models_to_download = {
    "bge-m3": "BAAI/bge-m3",
    "Qwen": "Qwen/Qwen3-VL-Embedding-2B",
    "embeddinggemma-300m": "google/embeddinggemma-300m",
    "embeddinggemma-2": "google/embeddinggemma-2",
    "sentence-transformers": "sentence-transformers/all-MiniLM-L6-v2"
}

# Patrones de archivos innecesarios a ignorar (duplicados ONNX, imágenes, etc.)
ignore = ["*.onnx*", "onnx/*", "imgs/*", "*.jpg", "*.jpeg", "*.webp", "*.png"]

# Descarga optimizada: solo los pesos esenciales, tokenizadores y configs
for name, model_id in models_to_download.items():
    target_dir = BASE_DIR / name
    print(f"Descargando {name} ({model_id}) en {target_dir}...")
    snapshot_download(
        repo_id=model_id,
        local_dir=str(target_dir),
        ignore_patterns=ignore,
        token=hf_token
    )
    print(f"{name} descargado en {target_dir}")


# Carga de modelos en ram

## Modelos disponibles
- bge-m3
- Qwen
- embeddinggemma-300m
- embeddinggemma-2
- sentence-transformers


In [ ]:
import sys
# Evita el conflicto de versión CUDA con torchaudio (innecesario para modelos de texto)
sys.modules['torchaudio'] = None

from sentence_transformers import SentenceTransformer

# 1. Elige el modelo que quieres cargar
model_name = "bge-m3"  # opciones: "bge-m3", "Qwen", "embeddinggemma-300m", "embeddinggemma-2", "sentence-transformers"

# Carga el modelo desde la carpeta local
model_path = f"/home/g5/reto_1_2-2627-Rag-Benchmark/Modelos/Embedding/{model_name}"
model = SentenceTransformer(model_path, trust_remote_code=True)

print(f"✓ {model_name} cargado correctamente en memoria")


In [ ]:
import sys
# Evita el conflicto de versión CUDA con torchaudio (innecesario para modelos de texto)
sys.modules['torchaudio'] = None

from sentence_transformers import SentenceTransformer

# 2. Elige los modelos que quieres cargar para el benchmark
selected_models = ["bge-m3", "embeddinggemma-300m","embeddinggemma-2", "sentence-transformers", "Qwen"]

models = {}
for name in selected_models:
    path = f"/home/g5/reto_1_2-2627-Rag-Benchmark/Modelos/Embedding/{name}"
    models[name] = SentenceTransformer(path, trust_remote_code=True)
    print(f"✓ {name} listo en memoria")


## Crear base de datos con cada modelo cargado



### 1. Chunking e indexación (BOE y Debian Handbook)
Cargamos y fragmentamos los documentos (BOE y Debian Handbook) respetando su estructura e indexamos los chunks en ChromaDB para cada modelo de embedding.

In [ ]:
import subprocess
import json
import chromadb

# Filtrar solo los modelos que no estén ya indexados en ChromaDB
client = chromadb.PersistentClient(path="./data/chroma")

documentos = [
    {"nombre": "BOE", "prefijo": "boe", "script": "../src/indexar_BOE.py"},
    {"nombre": "Debian Handbook", "prefijo": "debian", "script": "../src/indexar_debian.py"},
]

for doc in documentos:
    existentes = [c.name for c in client.list_collections()]
    modelos_a_indexar = [
        m for m in selected_models
        if f"{doc['prefijo']}_{m}" not in existentes or client.get_collection(f"{doc['prefijo']}_{m}").count() == 0
    ]

    if modelos_a_indexar:
        print(f"Indexando nuevos modelos ({doc['nombre']}): {modelos_a_indexar}")
        subprocess.run(
            [
                "python",
                doc["script"],
                json.dumps(modelos_a_indexar)
            ],
            check=True
        )
    else:
        print(f"Todos los modelos ({selected_models}) para {doc['nombre']} ya están indexados en ChromaDB.")


# Generacion de preguntas

In [ ]:
import gc, torch
# Liberar memoria VRAM ocupada por los modelos de embedding cargados en celdas previas
if "models" in globals():
    del models
if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from vllm import LLM, SamplingParams

# Carga del modelo K2 ya descargado con vLLM (ruta corregida)
llm = LLM(
    model="/home/g5/reto_1_2-2627-Rag-Benchmark/Modelos/Chat/K2-Horizon-3.7B-GGUF/K2-Horizon-4B-Q6_K.gguf",
    tokenizer="IFM/K2-Horizon-3.7B",
    hf_config_path="IFM/K2-Horizon-3.7B",
    trust_remote_code=True,
    max_model_len=2048,
    gpu_memory_utilization=0.4,
    enforce_eager=True,
)


In [ ]:
# Generación continua de preguntas aleatorias
import os, random, chromadb, pandas as pd
from vllm import SamplingParams

col = chromadb.PersistentClient(path="./data/chroma").list_collections()[0]
chunks = [(i, t) for i, t in enumerate(col.get()["documents"]) if len(t.strip()) > 80]
csv_path = "benchmark_embedding.csv"
df = pd.read_csv(csv_path) if os.path.exists(csv_path) else pd.DataFrame(columns=["texto", "pregunta", "solucion"])
params = SamplingParams(max_tokens=60, temperature=0.2, stop=["\n"])

print("Generando preguntas al azar... Pulsa Stop para parar.")
try:
    while True:
        idx, texto = random.choice(chunks)
        out = llm.generate([f"Texto: {texto}\nGenera una única pregunta sobre el texto anterior.\nPregunta: ¿"], params)
        pregunta = "¿" + out[0].outputs[0].text.split("\n")[0].split("?")[0].strip() + "?"
        
        fila = {"texto": texto, "pregunta": pregunta, "solucion": idx}
        for c in df.columns:
            if c not in fila:
                fila[c] = None
        
        df = pd.concat([df, pd.DataFrame([fila])], ignore_index=True)
        df.to_csv(csv_path, index=False)
        print(f"[{len(df)}] (Chunk {idx}): {pregunta}")
except KeyboardInterrupt:
    print(f"\nDetenido. Total preguntas: {len(df)}")


# Embedding

In [22]:
# Benchmark Completo de Embeddings (Recall@1, Recall@3, Recall@5, MRR, VRAM, Tiempo e Índice)
import os, gc, sys, time, torch, chromadb, pandas as pd
sys.modules['torchaudio'] = None
from sentence_transformers import SentenceTransformer
from IPython.display import display

csv_path = "benchmark_embedding.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"No se encontró {csv_path}. Ejecuta primero la generación de preguntas.")

df_preguntas = pd.read_csv(csv_path)

# Configuraciones de modelos con prefijos oficiales
model_configs = {
    "sentence-transformers": {"prefix": "", "desc": "sentence-transformers/all-MiniLM-L6-v2"},
    "embeddinggemma-300m": {"prefix": "task: search result | query: ", "desc": "google/embeddinggemma-300m"},
    "embeddinggemma-2": {"prefix": "task: search result | query: ", "desc": "google/embeddinggemma-2"},
    "bge-m3": {"prefix": "Represent this sentence for searching relevant passages: ", "desc": "BAAI/bge-m3"},
    "Qwen": {"prefix": "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: ", "desc": "Qwen/Qwen3-VL-Embedding-2B"}
}

client = chromadb.PersistentClient(path="./data/chroma")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Cargar colección de referencia para mapeo id -> índice
ref_col = client.get_collection(f"boe_{list(model_configs.keys())[0]}")
ref_ids = ref_col.get()["ids"]
id_to_idx = {doc_id: i for i, doc_id in enumerate(ref_ids)}
num_chunks = len(ref_ids)
chunk_texts = ref_col.get()["documents"]

resultados = []

for m, cfg in model_configs.items():
    col_name = f"boe_{m}"
    if col_name not in [c.name for c in client.list_collections()]:
        continue
    
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        gc.collect()
        
    model_path = f"/home/g5/reto_1_2-2627-Rag-Benchmark/Modelos/Embedding/{m}"
    model = SentenceTransformer(model_path, device=device, trust_remote_code=True)
    
    dim = model.get_embedding_dimension()
    params = sum(p.numel() for p in model.parameters())
    params_str = f"{params/1e6:.1f}M" if params < 1e9 else f"{params/1e9:.2f}B"
    
    # Medir tiempo de indexado
    t0 = time.perf_counter()
    _ = model.encode(chunk_texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    t_index = time.perf_counter() - t0
    
    # Medir VRAM pico
    vram_gb = round(torch.cuda.max_memory_allocated() / (1024**3), 2) if device == "cuda" else 0.0
    
    # Preparar queries con prefijo correspondiente
    queries = [f"{cfg['prefix']}{str(q).strip()}" if cfg["prefix"] and not str(q).strip().startswith(cfg["prefix"]) else str(q).strip() for q in df_preguntas["pregunta"]]
    q_embs = model.encode(queries, batch_size=64, normalize_embeddings=True, show_progress_bar=False).tolist()
    
    col = client.get_collection(col_name)
    n_total = len(df_preguntas)
    r1, r3, r5, rr_sum = 0, 0, 0, 0.0
    top1_predictions = []
    
    for i in range(n_total):
        sol_idx = int(df_preguntas.at[i, "solucion"])
        res = col.query(query_embeddings=[q_embs[i]], n_results=5)
        top_indices = [id_to_idx.get(doc_id, -1) for doc_id in res["ids"][0]]
        
        top1_predictions.append(top_indices[0])
        
        if sol_idx in top_indices[:1]: r1 += 1
        if sol_idx in top_indices[:3]: r3 += 1
        if sol_idx in top_indices[:5]:
            r5 += 1
            rank = top_indices[:5].index(sol_idx) + 1
            rr_sum += 1.0 / rank
            
    # Guardar predicciones de este modelo en el DataFrame
    df_preguntas[m] = top1_predictions
            
    idx_mb = round((num_chunks * dim * 4) / (1024 * 1024), 2)
    resultados.append({
        "Modelo": m,
        "Parámetros": params_str,
        "Dimensión": dim,
        "Recall@1": round(r1 / n_total, 4),
        "Recall@3": round(r3 / n_total, 4),
        "Recall@5": round(r5 / n_total, 4),
        "MRR": round(rr_sum / n_total, 4),
        "Indexado (s)": round(t_index, 2),
        "Índice (MB)": idx_mb,
        "VRAM (GB)": vram_gb
    })
    
    del model
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

# Guardar predicciones por modelo en benchmark_embedding.csv
df_preguntas.to_csv(csv_path, index=False)
print(f"✓ Predicciones por modelo guardadas en {csv_path}")

# Ordenar según criterio: 1) Recall@5 desc, 2) MRR desc, 3) VRAM asc
df_res = pd.DataFrame(resultados).sort_values(by=["Recall@5", "MRR", "VRAM (GB)"], ascending=[False, False, True]).reset_index(drop=True)
df_res.to_csv("resultados_benchmark_embedding.csv", index=False)

print("="*80)
print("Tabla A. Embeddings (Estatuto de los Trabajadores - BOE)")
print("="*80)
display(df_res)



Loading weights: 100%|██████████| 625/625 [00:00<00:00, 6600.78it/s]


✓ Predicciones por modelo guardadas en benchmark_embedding.csv
Tabla A. Embeddings (Estatuto de los Trabajadores - BOE)


,Modelo,Parámetros,Dimensión,Recall@1,Recall@3,Recall@5,MRR,Indexado (s),Índice (MB),VRAM (GB)
0,embeddinggemma-300m,307.6M,768,0.8224,0.9421,0.9710,0.8830,3.44,0.85,1.81
1,embeddinggemma-2,744.4M,768,0.7355,0.9131,0.9440,0.8228,1.60,0.85,4.40
2,Qwen,2.13B,2048,0.7490,0.9015,0.9305,0.8240,5.47,2.26,5.57
3,bge-m3,567.8M,1024,0.7259,0.8919,0.9305,0.8085,7.50,1.13,3.24
4,sentence-transformers,22.7M,384,0.3687,0.5656,0.6448,0.4717,0.45,0.42,0.37
